# AFib 탐지 튜토리얼 - MIMIC-IV-ECG 12-lead → 심방세동 이진분류

이 노트북은 **동봉된 두 파일**만 있으면 다른 스크립트 없이 단독 실행됩니다.
- `afib_data.csv` - 코호트 목록 (`subject_id`, `study_id`, `local_path`)
- `afib_concepts.csv` - AFib 진단(Condition) concept_id 목록 (OMOP CDM 진단 매칭용)

로컬/GCS DICOM 파형을 불러와 **환자 단위 6:2:2(train/val/test) 분할**로
심방세동(AFib) 분류기를 **한 번** 학습·검증합니다.

> **MI-CDM 철학**: 진단(AFib 여부)은 심전도 판독문을 보지 않고 **오직 OMOP CDM의
> `condition_occurrence`(진단 기록)로만** 판별합니다.

**데이터 경로 (GCS)**
- ECG : `gs://zarathu-edu-mimic-data-ethereal-mind-460209-e0/Datasets/ECG/open_datasets/MIMIC/mimic-iv-ecg-dcm/`
- CDM : `gs://zarathu-edu-mimic-data-ethereal-mind-460209-e0/Datasets/MIMIC-IV_CDM/`
- CXR : `gs://.../Datasets/MIMIC-CXR/files/` (본 노트북 미사용)

**전체 과정**
1. 코호트 목록(`afib_data.csv`) 로드
2. AFib 진단 encounter 준비 - `afib_concepts.csv` + CDM `condition_occurrence`
3. 전처리 함수 - 밴드패스 0.5-40Hz + 리드별 z-score
4. DICOM 파형 로드(+촬영시각) & 캐시(`.npz`)
4-1. 라벨 부여 - ECG 촬영일이 AFib encounter 구간에 들면 양성 (방문 단위)
5. 모델 정의 - 1D ResNet
6. 학습/평가 함수 준비
7. Baseline - 6:2:2 단일 분할 (직관적 첫 시도)
8. 엄밀성을 위해 - 5-fold 교차검증 준비
9~12. Step0 baseline -> best-val/early-stop -> augmentation -> seed ensemble (기법을 하나씩 얹기)
13. 리더보드 - 기법별 AUROC/AUPRC 상승 비교

> ⚠️ 방향성 확인용이며 프로덕션 벤치마크가 아닙니다.


## 0. 라이브러리 import & 설정

In [ ]:
# ---- 라이브러리 ----
import json, os, io
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
from scipy.signal import butter, filtfilt
import torch
import torch.nn as nn
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, confusion_matrix,
                             roc_curve, precision_recall_curve, f1_score, accuracy_score)
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# ==================== 실행 대상 ====================
USE_GCS = True          # True = GCS 버킷(운영) / False = 로컬 파일 테스트

# ==================== 데이터 경로 ====================
# --- GCS (운영) ---
GCS_ECG = "gs://zarathu-edu-mimic-data-ethereal-mind-460209-e0/Datasets/ECG/open_datasets/MIMIC/mimic-iv-ecg-dcm"
GCS_CDM = "gs://zarathu-edu-mimic-data-ethereal-mind-460209-e0/Datasets/MIMIC-IV_CDM"
GCS_CXR = "gs://zarathu-edu-mimic-data-ethereal-mind-460209-e0/Datasets/MIMIC-CXR/files"  # 미사용

# --- 로컬 (USE_GCS=False 일 때만 사용) ---
LOCAL_ROOT = Path("/Users/minseongkim/Desktop/youlab/mi-cdm")
LOCAL_CDM = LOCAL_ROOT / "MIMIC-IV-ECG/check_feasibility/data/MIMIC-IV-CDM/parquet"  # 로컬은 parquet

# --- 동봉 파일 (이 노트북과 같은 폴더; 다른 스크립트에 의존하지 않음) ---
HERE = Path.cwd()
DATA_CSV = HERE / "afib_data.csv"           # subject_id / study_id / local_path
CONCEPTS_CSV = HERE / "afib_concepts.csv"   # AFib 후보 concept_id
CACHE = HERE / "data_cache_tutorial.npz"    # 전처리된 파형 캐시
RESULT = HERE / "tutorial_result.json"      # 최종 지표 저장

# ==================== 하이퍼파라미터 ====================
SEED = 42
EPOCHS = 50                                   # 최대 에폭 (early stopping 으로 조기 종료 가능)
PATIENCE = 8                                 # val AUROC 가 이 에폭수만큼 안 오르면 조기 종료
BATCH = 32
LR = 1e-3
FS = 500.0                                   # 샘플링 주파수(Hz)

# device 우선순위: cuda -> mps -> cpu
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)

# ---- matplotlib 스타일 (깔끔하게) ----
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    pass
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 110,
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.labelsize": 11, "axes.edgecolor": "#888888",
    "axes.grid": True, "grid.alpha": 0.25, "legend.frameon": False,
})
# pandas 표 표시 옵션
pd.set_option("display.precision", 3)

# ---- pandas Styler(.style) 는 jinja2 필요 -> 없으면 설치 시도 (실패해도 아래 폴백으로 안전) ----
from IPython.display import display
try:
    import jinja2  # noqa: F401
except Exception:
    try:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "jinja2"])
    except Exception:
        pass

def disp_table(df, fmt=None, caption=None, gradient=None):
    # jinja2 있으면 Styler 로 예쁘게, 실패하면 반올림 DataFrame 으로 폴백 (절대 안 터짐)
    try:
        sty = df.style                             # jinja2 없으면 여기서 예외 -> 폴백
        if fmt is not None:
            sty = sty.format(fmt)
        if gradient is not None:
            sty = sty.background_gradient(cmap="Blues", subset=gradient)
        if caption is not None:
            sty = sty.set_caption(caption)
        return sty
    except Exception:
        out = df.copy()
        numc = out.select_dtypes("number").columns
        out[numc] = out[numc].round(3)
        if caption:
            print(caption)
        return out

print("USE_GCS =", USE_GCS, "| device =", DEVICE)


## 0-1. GCS 접근 준비 & 유틸리티

GCS를 쓸 경우 `fsspec`+`gcsfs`로 `gs://` 경로를 직접 읽습니다 (인증은 ADC/서비스계정 필요).
- DICOM 경로는 `afib_data.csv`의 `local_path` 컬럼을 그대로 사용합니다
  (형식: `.../files/p{앞4자리}/p{subject_id}/s{study_id}/{study_id}.dcm`)
- **ECG 촬영시각은 DICOM의 `AcquisitionDateTime` 에서 직접 읽습니다** (별도 컬럼/파일 불필요).
- CDM은 **헤더 있는 CSV**이므로 필요한 컬럼만 `usecols`로 읽습니다 (condition_occurrence 는 2.1GB → 청크 처리).

In [ ]:
# ---- GCS 라이브러리 준비 ----
fs = None
if USE_GCS:
    import importlib, subprocess, sys
    if importlib.util.find_spec("gcsfs") is None:
        print("gcsfs 설치 중...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gcsfs"])
    import fsspec, warnings
    fs = fsspec  # gs:// 및 로컬 모두 처리
    # google 라이브러리의 'Python 3.10 지원종료' FutureWarning 은 무해 -> 숨김
    warnings.filterwarnings("ignore", message="You are using a Python version",
                            category=FutureWarning)
    print("GCS 준비 완료")

# ---- DICOM 로드 (경로는 afib_data.csv 의 local_path 컬럼을 그대로 사용) ----
def load_waveform(path):
    # gs:// 경로면 fsspec, 아니면 로컬 파일로 DICOM 읽기
    # 반환: (파형 (5000,12), 촬영시각 Timestamp)  <- 촬영시각은 DICOM AcquisitionDateTime 에서 직접 읽음
    path = str(path)
    if path.startswith("gs://"):
        with fs.open(path, "rb") as f:
            ds = pydicom.dcmread(io.BytesIO(f.read()))
    else:
        ds = pydicom.dcmread(path)
    arr = ds.waveform_array(0)
    adt = getattr(ds, "AcquisitionDateTime", None)     # 예: '21800723084400'(YYYYMMDDHHMMSS)
    ecg_dt = pd.to_datetime(str(adt)[:14], format="%Y%m%d%H%M%S", errors="coerce") if adt else pd.NaT
    return arr, ecg_dt

def cdm_afib_encounters(afib_cids):
    # AFib 진단이 붙은 encounter(입원/내원) 구간: (subject_id, enc_start, enc_end)
    # MIMIC OMOP ETL 에서 condition_start_date ~ condition_end_date 가 그 진단이 내려진
    # stay 의 admit ~ discharge 에 해당 (MIMIC-IV-ECG-Ext-ICD 와 동일한 방문 단위 관점).
    # GCS: 헤더 있는 CSV(2.1GB) 를 청크로 읽어 필터 / 로컬: parquet
    cols = ["person_id", "condition_concept_id", "condition_start_date", "condition_end_date"]
    afib_set = set(afib_cids)
    frames = []
    keep = ["person_id", "condition_start_date", "condition_end_date"]
    if USE_GCS:
        CHUNK = 2_000_000
        # condition_occurrence 총 행 수 (MIMIC-IV CDM v5.3.1, 고정) -> 정확한 총 청크 수
        # (데이터셋이 바뀌면 이 값만 갱신하면 됨)
        CO_ROWS = 13_651_926
        n_chunks = int(np.ceil(CO_ROWS / CHUNK))                   # = 7
        reader = pd.read_csv(f"{GCS_CDM}/condition_occurrence.csv",
                             usecols=cols, chunksize=CHUNK)         # 헤더의 컬럼명으로 선택
        for chunk in tqdm(reader, total=n_chunks, desc="condition_occurrence 스캔",
                          unit="chunk"):
            cc = pd.to_numeric(chunk["condition_concept_id"], errors="coerce")
            frames.append(chunk.loc[cc.isin(afib_set), keep])
    else:
        co = pd.read_parquet(f"{LOCAL_CDM}/condition_occurrence.parquet", columns=cols)
        cc = pd.to_numeric(co["condition_concept_id"], errors="coerce")
        frames.append(co.loc[cc.isin(afib_set), keep])
    dx = pd.concat(frames, ignore_index=True)
    dx["person_id"] = pd.to_numeric(dx["person_id"], errors="coerce")
    dx["enc_start"] = pd.to_datetime(dx["condition_start_date"], errors="coerce").dt.normalize()
    dx["enc_end"] = pd.to_datetime(dx["condition_end_date"], errors="coerce").dt.normalize()
    dx["enc_end"] = dx["enc_end"].fillna(dx["enc_start"])          # end 결측이면 당일 취급
    dx = dx.dropna(subset=["person_id", "enc_start"])
    dx["person_id"] = dx["person_id"].astype("int64")
    dx = dx.merge(cdm_person_map(), on="person_id", how="inner")   # person_id -> subject_id
    return dx[["subject_id", "enc_start", "enc_end"]]

def cdm_person_map():
    # person_id <-> subject_id (person_source_value == subject_id)
    cols = ["person_id", "person_source_value"]
    if USE_GCS:
        pm = pd.read_csv(f"{GCS_CDM}/person.csv", usecols=cols)
    else:
        pm = pd.read_parquet(f"{LOCAL_CDM}/person.parquet", columns=cols)
    pm["person_id"] = pd.to_numeric(pm["person_id"], errors="coerce")
    pm["subject_id"] = pd.to_numeric(pm["person_source_value"], errors="coerce")
    pm = pm.dropna(subset=["person_id", "subject_id"])
    pm = pm.astype({"person_id": "int64", "subject_id": "int64"})
    return pm[["person_id", "subject_id"]]


## 1. 코호트 목록 로드 (`afib_data.csv`)

In [ ]:
# subject_id / study_id / local_path  (ECG 촬영시각은 4단계에서 DICOM 에서 직접 읽음)
data = pd.read_csv(DATA_CSV)
print("행(ECG) 수:", len(data), "/ 고유 환자:", data["subject_id"].nunique())
data.head()


## 2. AFib 진단 encounter 준비 - CDM `condition_occurrence` (방문 단위)

**MI-CDM 철학**: 심전도 판독문은 보지 않고 CDM 진단 기록으로만 AFib 여부를 판별합니다.
매칭은 **방문(encounter, 입원/내원) 단위** - MIMIC-IV-ECG의 표준 라벨셋 **MIMIC-IV-ECG-Ext-ICD**
와 동일한 관점입니다. ECG가 기록된 stay의 진단에 AFib가 있으면 그 ECG를 양성으로 봅니다.

1. `afib_concepts.csv`(동봉)에서 AFib 진단(Condition) concept 집합을 가져옴
2. CDM `condition_occurrence`에서 그 진단의 (`person_id`, stay 구간=`condition_start_date`~`condition_end_date`)
   - MIMIC OMOP ETL에서 이 구간이 그 진단이 내려진 입원/내원의 admit~discharge에 해당
3. CDM `person`으로 `person_id` → `subject_id`(=`person_source_value`) 매핑

> 여기서는 AFib 진단 encounter 구간(`enc`)만 준비합니다.
> **실제 ECG별 라벨 부여는 4단계(DICOM에서 촬영시각을 읽은 뒤) 다음의 4-1단계**에서 합니다.
> 이렇게 하는 근거는 노트북 맨 끝 **References** 참고.
> (GCS면 `condition_occurrence.csv` 2.1GB를 스캔하므로 몇 분 걸릴 수 있습니다.)

In [ ]:
# 1) AFib 후보 concept (동봉 파일 - 다른 스크립트에 의존하지 않음)
afib_cids = pd.read_csv(CONCEPTS_CSV)["concept_id"].unique().tolist()
print("AFib 후보 concept_id 수:", len(afib_cids))

# 2~3) AFib 진단이 붙은 encounter(입원/내원) 구간 (subject_id, enc_start, enc_end)
enc = cdm_afib_encounters(afib_cids)
print("AFib 진단 encounter:", len(enc), "| 진단 환자:", enc["subject_id"].nunique())
enc.head()


## 3. 전처리 함수 정의

각 파형은 (5000 samples × 12 leads).
- **밴드패스 0.5-40Hz** (4차 버터워스, `filtfilt` 무위상) - 기저선 변동·고주파 잡음 제거
- **리드별 z-score** - 리드마다 평균 0·표준편차 1
- 최종 shape: (12, 5000) float32

In [ ]:
def bandpass(sig, lo=0.5, hi=40.0, fs=FS, order=4):
    b, a = butter(order, [lo / (fs / 2), hi / (fs / 2)], btype="band")
    return filtfilt(b, a, sig, axis=-1)

def preprocess(arr):
    # arr: (5000, 12) -> (12, 5000) 밴드패스 + 리드별 z-score
    x = arr.T.astype(np.float64)
    x = bandpass(x)
    mu = x.mean(axis=1, keepdims=True)
    sd = x.std(axis=1, keepdims=True) + 1e-6
    x = (x - mu) / sd
    return x.astype(np.float32)


## 4. DICOM 파형 로드 & 캐시 생성

각 ECG의 DICOM을 읽어 전처리한 파형 `X`, `subject`, `study`, 그리고 **촬영시각 `ecg_date`
(DICOM `AcquisitionDateTime`)** 를 만듭니다. 라벨 `y`는 캐시하지 않고 다음 4-1단계에서 계산합니다
(진단 정의를 바꿔도 파형을 다시 안 읽어도 됨). 한 번 만든 파형은 `.npz`로 캐시합니다
(형식 이상·NaN 파일은 건너뜀).

In [ ]:
if CACHE.exists():
    d = np.load(CACHE, allow_pickle=False)
    X, subject, study = d["X"], d["subject"], d["study"]
    ecg_date = pd.to_datetime(d["ecg_date"])          # datetime64[ns]
    print(f"[캐시 로드] N={len(X)}  ({CACHE.name})")
else:
    rows = data[["subject_id", "study_id", "local_path"]].to_dict("records")
    X, subj, study, ecg_dt = [], [], [], []
    skipped = 0
    for r in tqdm(rows, desc="DICOM 로드", unit="ecg"):
        path = r["local_path"]                        # afib_data.csv 의 경로 그대로 사용
        try:
            arr, dt = load_waveform(path)             # 파형 (5000,12) + 촬영시각
            if arr.shape != (5000, 12) or np.isnan(arr).any():
                skipped += 1
                continue
            X.append(preprocess(arr))
            subj.append(int(r["subject_id"]))
            study.append(int(r["study_id"]))
            ecg_dt.append(dt)
        except Exception as e:
            skipped += 1
            if skipped <= 10:
                tqdm.write(f"skip {r['study_id']} {e}")
    print(f"로드 완료 (skip={skipped})")
    X = np.stack(X)
    subject = np.array(subj)
    study = np.array(study)
    ecg_date = pd.to_datetime(pd.Series(ecg_dt))
    np.savez_compressed(CACHE, X=X, subject=subject, study=study,
                        ecg_date=ecg_date.values.astype("datetime64[ns]"))
    print(f"\n[캐시 저장] {CACHE}  |  N={len(X)}  환자={len(set(subject))}  skipped={skipped}")

print("X shape:", X.shape, "| 촬영시각 결측:", int(pd.isna(ecg_date).sum()))


## 4-1. 라벨 부여 (방문 단위 매칭)

로드된 각 ECG의 **촬영일(`ecg_date`)** 이 그 환자의 AFib encounter 구간(2단계 `enc`) 안에
들어가면 그 ECG를 양성(1)으로 라벨링합니다. (판독문 미사용, 방문 단위)

In [ ]:
lab = pd.DataFrame({"study_id": study, "subject_id": subject})
lab["ecg_date"] = pd.to_datetime(np.asarray(ecg_date)).normalize()   # 촬영일(자정 기준)
m = lab.merge(enc, on="subject_id", how="left")
inside = (m["ecg_date"] >= m["enc_start"]) & (m["ecg_date"] <= m["enc_end"])
pos_studies = set(m.loc[inside.fillna(False), "study_id"])
y = np.array([1 if s in pos_studies else 0 for s in study], dtype=int)
print(f"[encounter] AFib+ = {int(y.sum())} / {len(y)}  ({100*y.mean():.1f}%)  "
      f"| 양성 환자={len(set(subject[y == 1]))}")


## 5. 모델 정의 - 1D ResNet

In [ ]:
class ResBlock1D(nn.Module):
    # 1D 잔차 블록 (Conv-BN-ReLU x2 + skip connection)
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(cin, cout, 7, stride=stride, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(cout)
        self.conv2 = nn.Conv1d(cout, cout, 7, padding=3, bias=False)
        self.bn2 = nn.BatchNorm1d(cout)
        self.act = nn.ReLU(inplace=True)
        self.down = None
        if stride != 1 or cin != cout:
            self.down = nn.Sequential(
                nn.Conv1d(cin, cout, 1, stride=stride, bias=False),
                nn.BatchNorm1d(cout))

    def forward(self, x):
        idt = x if self.down is None else self.down(x)
        out = self.act(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.act(out + idt)


class ResNet1D(nn.Module):
    # 12-lead 입력 -> 1 logit 출력 (이진분류)
    def __init__(self, cin=12):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(cin, 32, 15, stride=2, padding=7, bias=False),
            nn.BatchNorm1d(32), nn.ReLU(inplace=True), nn.MaxPool1d(3, 2, 1))
        self.layers = nn.Sequential(
            ResBlock1D(32, 32), ResBlock1D(32, 64, stride=2),
            ResBlock1D(64, 128, stride=2), ResBlock1D(128, 128, stride=2))
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Dropout(0.3), nn.Linear(128, 1))

    def forward(self, x):
        return self.head(self.layers(self.stem(x))).squeeze(1)


## 6. 학습·평가 함수 준비

먼저 이후 모든 단계에서 재사용할 **학습·평가 함수**를 정의합니다.

- `fit_model(...)`: 한 번 학습. `best_val` / `early_stop` / `augment` 플래그로
  기법을 켜고 끕니다 (Step마다 하나씩 켤 예정).
- `predict / eval_oof / show_step`: 예측 확률, 지표 계산(AUROC/AUPRC + 운영점), 표·곡선 표시.

`fit_model`은 진행바(`pbar`)를 받아 **epoch 단위로** 갱신하므로, 뒤의 5-fold CV에서
전체 학습 진행률이 한 줄로 매끄럽게 나옵니다.

In [ ]:
TARGET = 0.90            # high-sens / high-spec 목표치
N_SPLITS = 5

# ---- 공통 유틸 ----
def predict(model, Xe):
    model.eval()
    with torch.no_grad():
        Xe_t = torch.tensor(Xe, device=DEVICE)
        logits = torch.cat([model(Xe_t[i:i + 64]) for i in range(0, len(Xe_t), 64)])
    return torch.sigmoid(logits).cpu().numpy()

def evaluate(model, Xe, ye):                # 학습 중 val AUROC 용
    p = predict(model, Xe)
    return roc_auc_score(ye, p) if len(set(ye)) > 1 else float("nan")

def augment_batch(xb):
    # ECG 특화 augmentation (Step 3에서 사용): 진폭 스케일 + 잡음 + 시간 이동 + 리드 마스킹
    B = xb.shape[0]
    xb = xb * torch.empty(B, 1, 1, device=xb.device).uniform_(0.8, 1.2)    # 진폭 스케일
    xb = xb + 0.02 * torch.randn_like(xb)                                  # 가우시안 잡음
    xb = torch.roll(xb, shifts=int(torch.randint(-100, 101, (1,))), dims=-1)  # 시간 이동
    k = int(torch.randint(0, 3, (1,)))                                     # 리드 0~2개 마스킹
    if k > 0:
        xb = xb.clone()
        xb[:, torch.randperm(12, device=xb.device)[:k], :] = 0.0
    return xb

# ---- 한 fold 학습 (기법은 플래그로 on/off) ----
import copy
def fit_model(train_idx, val_idx, seed=SEED, pbar=None, pbar_info=None,
              best_val=False, early_stop=False, scheduler=False, augment=False):
    torch.manual_seed(seed)
    model = ResNet1D().to(DEVICE)
    ytr = y[train_idx]
    pos_w = torch.tensor([(len(ytr) - ytr.sum()) / max(ytr.sum(), 1)],
                         dtype=torch.float32, device=DEVICE)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = (torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=4)
             if scheduler else None)
    Xtr_t = torch.tensor(X[train_idx], device=DEVICE)
    ytr_t = torch.tensor(ytr.astype("float32"), device=DEVICE)
    n = len(train_idx)
    own_bar = pbar is None
    if own_bar:                                       # 단독 호출(baseline)이면 자체 진행바
        pbar = tqdm(total=EPOCHS, desc="학습", unit="epoch")
    best_au, best_state, wait = -1.0, None, 0
    for ep in range(EPOCHS):
        model.train()
        perm = torch.randperm(n)
        for i in range(0, n, BATCH):
            idx = perm[i:i + BATCH]
            xb = Xtr_t[idx]
            if augment:
                xb = augment_batch(xb)
            opt.zero_grad()
            loss = crit(model(xb), ytr_t[idx])
            loss.backward(); opt.step()
        va = None
        if best_val or early_stop or scheduler:      # val 평가가 필요한 기법이 켜졌을 때만
            va = evaluate(model, X[val_idx], y[val_idx])
            if sched is not None:
                sched.step(va)
            if va > best_au:
                best_au, best_state, wait = va, copy.deepcopy(model.state_dict()), 0
            else:
                wait += 1
        # 진행바: epoch 단위로 갱신 (5-fold CV 전체가 한 줄로 매끄럽게 진행)
        post = {}
        if pbar_info:
            post["단계"] = pbar_info
        if va is not None:
            post["val_auroc"] = f"{va:.3f}"
        if post:
            pbar.set_postfix(post)
        pbar.update(1)
        if early_stop and va is not None and wait >= PATIENCE:
            pbar.update(EPOCHS - 1 - ep)             # 남은 epoch만큼 채워 전체 진행률 유지
            break
    if own_bar:
        pbar.close()
    if best_val and best_state is not None:
        model.load_state_dict(best_state)            # best-val 가중치 복원
    return model

# ---- 예측/지표 평가 유틸 (단일 분할 · CV 공용) ----
FLOAT_COLS = ["threshold", "accuracy", "sensitivity", "specificity", "precision", "f1"]
FMT = {c: "{:.3f}" for c in FLOAT_COLS}
OP_YOUDEN, OP_HS, OP_SP = "Youden", f"High-Sens(>={TARGET:.2f})", f"High-Spec(>={TARGET:.2f})"

def metrics_row(name, thr, proba, ytrue):
    pred = (proba >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(ytrue, pred, labels=[0, 1]).ravel()
    return {"operating_point": name, "threshold": float(thr),
            "accuracy": float(accuracy_score(ytrue, pred)),
            "sensitivity": tp / (tp + fn + 1e-9), "specificity": tn / (tn + fp + 1e-9),
            "precision": tp / (tp + fp + 1e-9), "f1": float(f1_score(ytrue, pred, zero_division=0)),
            "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn)}

def eval_oof(oof):
    m = ~np.isnan(oof)
    yv, pv = y[m].astype(int), oof[m]
    auroc, auprc = roc_auc_score(yv, pv), average_precision_score(yv, pv)
    fpr, tpr, thr = roc_curve(yv, pv); sens, spec = tpr, 1 - fpr
    iy = int(np.argmax(sens - fpr))
    chs = np.where(sens >= TARGET)[0]; ihs = int(chs[np.argmax(spec[chs])]) if len(chs) else iy
    csp = np.where(spec >= TARGET)[0]; isp = int(csp[np.argmax(sens[csp])]) if len(csp) else iy
    ops = {OP_YOUDEN: thr[iy], OP_HS: thr[ihs], OP_SP: thr[isp]}
    df = pd.DataFrame([metrics_row(nm, t, pv, yv) for nm, t in ops.items()]).set_index("operating_point")
    return {"y": yv, "proba": pv, "auroc": float(auroc), "auprc": float(auprc), "df": df}

def show_step(res, title):
    print(f"[{title}]  AUROC = {res['auroc']:.3f}   AUPRC = {res['auprc']:.3f}")
    display(disp_table(res["df"][FLOAT_COLS], fmt=FMT,
            gradient=["sensitivity", "specificity", "precision", "f1"],
            caption=f"{title} - 운영점별 지표"))
    yv, pv, df = res["y"], res["proba"], res["df"]
    fpr, tpr, _ = roc_curve(yv, pv); prec, rec, _ = precision_recall_curve(yv, pv)
    fig, ax = plt.subplots(1, 2, figsize=(11, 5))
    ax[0].plot(fpr, tpr, lw=2.2, color="#2563eb", label=f"AUROC={res['auroc']:.3f}")
    ax[0].fill_between(fpr, tpr, alpha=0.08, color="#2563eb")
    ax[0].plot([0, 1], [0, 1], "--", lw=1, color="#9ca3af")
    for nm, c, mk in [(OP_YOUDEN, "#ef4444", "o"), (OP_HS, "#10b981", "s"), (OP_SP, "#8b5cf6", "^")]:
        r = df.loc[nm]
        ax[0].scatter(r["fp"] / (r["fp"] + r["tn"] + 1e-9), r["tp"] / (r["tp"] + r["fn"] + 1e-9),
                      s=80, marker=mk, color=c, edgecolor="white", zorder=5, label=nm)
    ax[0].set(xlim=(-0.02, 1.02), ylim=(-0.02, 1.02), xlabel="1 - Specificity",
              ylabel="Sensitivity", title=f"ROC - {title}"); ax[0].legend(loc="lower right", fontsize=8)
    ax[1].plot(rec, prec, lw=2.2, color="#0891b2", label=f"AUPRC={res['auprc']:.3f}")
    ax[1].fill_between(rec, prec, alpha=0.08, color="#0891b2")
    ax[1].axhline(float(yv.mean()), ls="--", lw=1, color="#9ca3af", label=f"baseline={yv.mean():.3f}")
    ax[1].set(xlim=(-0.02, 1.02), ylim=(-0.02, 1.02), xlabel="Recall", ylabel="Precision",
              title=f"PR - {title}"); ax[1].legend(loc="lower left", fontsize=8)
    plt.tight_layout(); plt.show()

print("공용 함수 준비 완료")


## 7. Baseline - 6:2:2 단일 분할 (환자 단위)

가장 직관적인 첫 시도부터 봅니다. `StratifiedGroupKFold`로 만든 5조각을
**train : val : test = 3 : 1 : 1 (= 60 : 20 : 20)** 로 나눠 **한 번만** 학습·평가합니다.
(환자 단위라 같은 환자가 여러 세트에 걸치지 않음 -> 누수 없음)

빠르지만 **어느 환자가 test에 들어가느냐에 따라 성능이 출렁**입니다.
이 한계가 다음 단계에서 5-fold CV로 넘어가는 이유가 됩니다.

In [ ]:
# 6:2:2 단일 분할 (환자 단위)
_folds = [te for _, te in
          StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(X, y, groups=subject)]
ho_test, ho_val = _folds[0], _folds[1]
ho_train = np.concatenate(_folds[2:])
for nm, idx in [("train", ho_train), ("val", ho_val), ("test", ho_test)]:
    print(f"{nm:5s}: n={len(idx):4d}  AFib+={int(y[idx].sum()):4d} "
          f"({100*y[idx].mean():.1f}%)  환자={len(set(subject[idx]))}")

# 아무 기법 없이 학습 (마지막 epoch) -> test 로 평가
_model = fit_model(ho_train, ho_val)
_oof = np.full(len(y), np.nan); _oof[ho_test] = predict(_model, X[ho_test])
res_holdout = eval_oof(_oof)
show_step(res_holdout, "Baseline (6:2:2 단일 분할)")


## 8. 엄밀성을 위해 - 5-fold 교차검증 준비

단일 분할은 test에 걸린 환자에 따라 성능이 흔들립니다. 그래서 **5-fold 교차검증(CV)** 으로 넘어갑니다.

- `StratifiedGroupKFold`로 **환자 단위** 5조각(fold 0~4)을 만듭니다 (**같은 환자는 한 조각에만**).
- fold를 하나씩 test로 돌리며, 각 fold를 **train : val : test = 3 : 1 : 1** 로 나눕니다.

| k (test) | test | val | train |
|:---:|:---:|:---:|:---:|
| 0 | 조각 0 | 조각 1 | 조각 2,3,4 |
| 1 | 조각 1 | 조각 2 | 조각 3,4,0 |
| 2 | 조각 2 | 조각 3 | 조각 4,0,1 |
| 3 | 조각 3 | 조각 4 | 조각 0,1,2 |
| 4 | 조각 4 | 조각 0 | 조각 1,2,3 |

- 5개 fold의 test 예측을 모두 모으면(**pooled out-of-fold**) 모든 ECG가 정확히 한 번씩
  test로 평가됩니다 -> 단일 분할보다 훨씬 견고. **최종 성능은 이 pooled OOF로만** 측정합니다.
- 진행바는 fold 5개 × epoch 전체에 대해 **한 줄로 매끄럽게** 표시됩니다.

In [ ]:
FOLDS = [te for _, te in
         StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED).split(X, y, groups=subject)]

def run_cv(seeds=(SEED,), desc="CV", **cfg):
    oof = np.full(len(y), np.nan)
    total = N_SPLITS * len(seeds) * EPOCHS               # 전체 학습량(=진행바 total)
    pbar = tqdm(total=total, desc=desc, unit="epoch")
    for k in range(N_SPLITS):
        test_idx = FOLDS[k]
        val_idx = FOLDS[(k + 1) % N_SPLITS]
        train_idx = np.concatenate([FOLDS[j] for j in range(N_SPLITS)
                                    if j not in (k, (k + 1) % N_SPLITS)])
        probs = []
        for si, sd in enumerate(seeds):
            info = f"fold {k+1}/{N_SPLITS}" + (f" seed {si+1}/{len(seeds)}" if len(seeds) > 1 else "")
            m = fit_model(train_idx, val_idx, seed=sd, pbar=pbar, pbar_info=info, **cfg)
            probs.append(predict(m, X[test_idx]))
        oof[test_idx] = np.mean(probs, axis=0)          # seed 여러개면 앙상블 평균
    pbar.close()
    return oof

board = {}   # Step별 결과 리더보드 누적
print("5-fold CV 준비 완료 | folds:", [len(f) for f in FOLDS])


## 9. Step 0 - CV Baseline (기법 없음)

같은 5-fold CV 틀에서, 아무 기법 없이 고정 `EPOCHS`만큼 학습하고 **마지막 epoch** 가중치로 평가합니다.
리더보드의 **출발점**입니다.

In [ ]:
oof0 = run_cv(desc="Step0 baseline")
res0 = eval_oof(oof0); board["Step0 baseline"] = res0
show_step(res0, "Step0 baseline (CV)")


## 10. Step 1 - best-val + early stopping

매 epoch val AUROC를 보고 **가장 좋았던 시점의 가중치**를 복원합니다 (마지막 epoch가 아니라).
개선이 `PATIENCE` epoch 동안 없으면 조기 종료 -> 과적합 방지 + 시간 절약.

In [ ]:
oof1 = run_cv(desc="Step1 best-val", best_val=True, early_stop=True)
res1 = eval_oof(oof1); board["Step1 +best-val/early-stop"] = res1
show_step(res1, "Step1 +best-val/early-stop")


## 11. Step 2 - + Data augmentation

학습 배치마다 ECG를 살짝 변형(진폭 스케일 · 가우시안 잡음 · 시간 이동 · 리드 마스킹)해
**데이터 다양성**을 늘립니다. 모델이 특정 패턴에 과적합되지 않고 일반화되도록 돕습니다.

목표는 **데이터 양이 아니라 경험의 다양성**입니다. 여러 상황을 겪어 본 모델은 처음 보는
심전도에도 강건합니다.

> 참고: Wen et al., *Time Series Data Augmentation for Deep Learning - A Survey*, 2020 (arXiv:2002.12478)

### 네 가지 변형

**① 진폭 스케일** - 사람·기계마다 파형 크기가 다릅니다. **크기가 아니라 모양**에 집중하도록 유도.
- 방법: 파형 전체 진폭에 배수(0.8~1.2배) 곱하기
- 효과: 신호 크기와 무관하게 같은 병으로 인식

**② 가우시안 잡음** - 실제 병원 심전도엔 근육 떨림·전극 잡음이 상존합니다. **실전 잡음을 미리 겪게** 함.
- 방법: 파형에 아주 작은 무작위 잡음 추가
- 효과: 실제 환경 잡음에도 판단 안정

**③ 시간 이동** - 심전도를 어느 순간부터 잘랐는지는 사실상 우연입니다. **위치는 우연, 모양이 본질**.
- 방법: 파형 전체를 시간축으로 조금 이동
- 효과: 시작 지점과 무관하게 같은 패턴으로 인식

**④ 리드 마스킹** - 한 리드에만 의존하면 그 리드가 불량일 때 판단이 붕괴합니다. **한 곳에 기대지 않기**.
- 방법: 12리드 중 0~2개를 무작위로 가린 채 학습
- 효과: 특정 리드 의존 감소, 판단 안정

아래에서 **실제로 파형이 어떻게 바뀌는지** 원본과 비교해 보여줍니다 (한 ECG 예시).

In [ ]:
# ---- augmentation 시각화: 원본 vs 각 변형 (양성 ECG 하나, lead II) ----
_i = int(np.where(y == 1)[0][0])            # 양성 ECG 하나
sig = X[_i]                                 # (12, 5000), 전처리(z-score)된 파형
lead, t = 1, np.arange(X.shape[2]) / FS     # lead II, 시간축(초)

def _one(kind):                             # 개별 변형 하나만 적용해 효과를 명확히
    xt = torch.tensor(sig[None].copy(), device=DEVICE)
    if kind == "scale":
        xt = xt * 1.2                       # 진폭 1.2배
    elif kind == "noise":
        torch.manual_seed(0); xt = xt + 0.03 * torch.randn_like(xt)
    elif kind == "shift":
        xt = torch.roll(xt, shifts=100, dims=-1)   # +0.2초 이동
    return xt[0].cpu().numpy()

fig, ax = plt.subplots(4, 1, figsize=(11, 8), sharex=True)
ax[0].plot(t, sig[lead], color="#111827", lw=0.8); ax[0].set_title("원본 (lead II)")
ax[1].plot(t, sig[lead], color="#d1d5db", lw=0.8, label="원본")
ax[1].plot(t, _one("scale")[lead], color="#2563eb", lw=0.8, label="변형")
ax[1].set_title("① 진폭 스케일 (x1.2)"); ax[1].legend(loc="upper right", fontsize=8)
ax[2].plot(t, sig[lead], color="#d1d5db", lw=0.8)
ax[2].plot(t, _one("noise")[lead], color="#059669", lw=0.8); ax[2].set_title("② 가우시안 잡음")
ax[3].plot(t, sig[lead], color="#d1d5db", lw=0.8)
ax[3].plot(t, _one("shift")[lead], color="#7c3aed", lw=0.8); ax[3].set_title("③ 시간 이동 (+0.2s)")
ax[3].set_xlabel("time (s)")
for a in ax:
    a.set_ylabel("z-score")
plt.tight_layout(); plt.show()

# ---- ④ 리드 마스킹: 12리드 heatmap, 일부 리드를 0으로 ----
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
kw = dict(aspect="auto", cmap="RdBu_r", vmin=-3, vmax=3)
ax[0].imshow(sig, **kw); ax[0].set_title("원본 12-lead"); ax[0].set_ylabel("lead"); ax[0].set_xlabel("sample")
masked = sig.copy(); masked[[3, 8]] = 0.0
ax[1].imshow(masked, **kw); ax[1].set_title("④ 리드 마스킹 (2개 리드 0)"); ax[1].set_xlabel("sample")
plt.tight_layout(); plt.show()

print("실제 학습에선 위 네 변형이 배치마다 무작위 조합으로 한 번에 적용됩니다 (augment_batch).")


In [ ]:
oof2 = run_cv(desc="Step2 +aug", best_val=True, early_stop=True, augment=True)
res2 = eval_oof(oof2); board["Step2 +augmentation"] = res2
show_step(res2, "Step2 +augmentation")


## 12. Step 3 - + Seed ensemble

동일한 설정으로 **seed만 바꿔 여러 번(여기선 3개) 학습**한 뒤, fold마다 예측 확률을 평균냅니다.
한 번의 학습에 실린 **운(초기화·배치 순서의 무작위성)을 상쇄**해 분산을 줄이고 안정적으로 성능을 끌어올리는,
가장 흔한 마무리 기법입니다.

- 방법: seed 3개로 각각 학습 → 같은 test fold에 대한 예측 확률을 산술 평균
- 효과: 개별 모델의 분산 감소 → AUROC/AUPRC가 안정적으로 상승

> 참고: Lakshminarayanan et al., *Simple and Scalable Predictive Uncertainty Estimation using Deep Ensembles*, NeurIPS 2017 (arXiv:1612.01474)

In [ ]:
oof3 = run_cv(seeds=(SEED, SEED + 1, SEED + 2), desc="Step3 +ensemble",
              best_val=True, early_stop=True, augment=True)
res3 = eval_oof(oof3); board["Step3 +seed ensemble"] = res3
show_step(res3, "Step3 +seed ensemble")


## 13. 리더보드 - 기법을 얹을수록 오르는 성능

각 Step에서 기법을 하나씩 더할 때마다 **pooled AUROC/AUPRC**가 어떻게 변하는지 비교합니다.
(모두 동일한 5-fold CV 틀에서 평가했으므로 공정한 비교입니다.)

In [ ]:
lb = pd.DataFrame({k: [v["auroc"], v["auprc"]] for k, v in board.items()},
                  index=["AUROC", "AUPRC"]).T
display(disp_table(lb, fmt="{:.3f}", gradient=["AUROC", "AUPRC"],
                   caption="Step별 성능 (5-fold CV pooled)"))

# 상승 곡선
fig, ax = plt.subplots(figsize=(8.5, 4.6))
xs = list(range(len(lb)))
ax.plot(xs, lb["AUROC"], "-o", lw=2.2, color="#2563eb", label="AUROC")
ax.plot(xs, lb["AUPRC"], "-s", lw=2.2, color="#0891b2", label="AUPRC")
for i, (a, p) in enumerate(zip(lb["AUROC"], lb["AUPRC"])):
    ax.annotate(f"{a:.3f}", (i, a), textcoords="offset points", xytext=(0, 9), ha="center", fontsize=8)
    ax.annotate(f"{p:.3f}", (i, p), textcoords="offset points", xytext=(0, -15), ha="center", fontsize=8)
ax.set_xticks(xs); ax.set_xticklabels(lb.index, rotation=18, ha="right", fontsize=8)
ax.set_ylabel("score"); ax.set_title("기법을 얹을수록 오르는 성능 (5-fold CV pooled)")
ax.legend(loc="lower right"); plt.tight_layout(); plt.show()

# ---- Step별 ROC / PR 곡선 오버레이 (곡선이 좌상단으로 밀리는 = 성능 상승) ----
steps = list(board.keys())
cmap = plt.cm.viridis(np.linspace(0.15, 0.85, len(steps)))
fig, ax = plt.subplots(1, 2, figsize=(12, 5.2))
for c, name in zip(cmap, steps):
    r = board[name]
    fpr, tpr, _ = roc_curve(r["y"], r["proba"])
    prec, rec, _ = precision_recall_curve(r["y"], r["proba"])
    lw = 2.6 if name == steps[-1] else 1.6          # 최종 Step 강조
    ax[0].plot(fpr, tpr, color=c, lw=lw, label=f"{name} ({r['auroc']:.3f})")
    ax[1].plot(rec, prec, color=c, lw=lw, label=f"{name} ({r['auprc']:.3f})")
ax[0].plot([0, 1], [0, 1], "--", lw=1, color="#9ca3af")
ax[0].set(xlim=(-0.02, 1.02), ylim=(-0.02, 1.02), xlabel="1 - Specificity", ylabel="Sensitivity",
          title="ROC - Step별 (AUROC)"); ax[0].legend(loc="lower right", fontsize=8)
base = float(board[steps[-1]]["y"].mean())
ax[1].axhline(base, ls="--", lw=1, color="#9ca3af", label=f"baseline={base:.2f}")
ax[1].set(xlim=(-0.02, 1.02), ylim=(-0.02, 1.02), xlabel="Recall", ylabel="Precision",
          title="Precision-Recall - Step별 (AUPRC)"); ax[1].legend(loc="lower left", fontsize=8)
plt.tight_layout(); plt.show()

# ---- 최종 모델 성능 강조 카드 ----
best = board[steps[-1]]
fig, ax = plt.subplots(1, 2, figsize=(9, 2.6))
for a, (label, val, color) in zip(ax, [("AUROC", best["auroc"], "#2563eb"),
                                       ("AUPRC", best["auprc"], "#0891b2")]):
    a.barh([0], [val], color=color, height=0.5)
    a.barh([0], [1], color="#e5e7eb", height=0.5, zorder=0)
    a.text(0.02, 0, f"{label}  {val:.3f}", va="center", ha="left", fontsize=15,
           fontweight="bold", color="white")
    a.set_xlim(0, 1); a.axis("off")
fig.suptitle(f"최종 모델 ({steps[-1]}) 성능", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()

# 결과 저장
with open(RESULT, "w") as f:
    json.dump({"target": TARGET, "n_splits": N_SPLITS,
               "leaderboard": {k: {"auroc": round(v["auroc"], 4), "auprc": round(v["auprc"], 4)}
                               for k, v in board.items()}}, f, indent=2, ensure_ascii=False)
print("저장:", RESULT)


## 마무리

- 이 노트북은 동봉된 `afib_data.csv` + `afib_concepts.csv` 만으로 단독 실행됩니다 (다른 스크립트 의존 X).
- 평가는 **환자 단위 5-fold 교차검증(pooled OOF)** 으로, 모든 ECG가 정확히 한 번씩 test로 쓰여 견고합니다.
- Step0(baseline) -> best-val/early-stop -> augmentation -> seed ensemble 순으로 **기법을 하나씩 얹을수록 성능이 오르는** 과정을 리더보드로 보여줍니다.
- 운영점(Youden / high-sensitivity / high-specificity) 임계값은 pooled OOF에서 잡습니다 (교육용 단순화).
- 진단(AFib)은 **오직 CDM `condition_occurrence`** 로 판별하며, **방문(encounter, 입원/내원) 단위**로 매칭합니다 (MI-CDM 철학, 판독문 미사용). 근거는 아래 References.
- ECG 촬영시각은 별도 컬럼 없이 **DICOM `AcquisitionDateTime` 에서 직접** 읽습니다 (`afib_data.csv`는 3컬럼).
- 재실행 시 `data_cache_tutorial.npz`(파형+촬영시각) 덕분에 DICOM 로드를 건너뜁니다. 라벨은 매번 4-1에서
  다시 계산하므로 진단 정의만 바꿀 땐 캐시 삭제 불필요 (경로·전처리·코호트를 바꾸면 캐시 삭제).
- GCS 실행 시 인증(ADC/서비스계정)과 `gcsfs`가 필요합니다.


## References - 라벨링(ECG ↔ AFib 진단 매칭) 근거

**왜 방문(encounter, 입원/내원) 단위로 매칭하나?**

MIMIC-IV-ECG의 **표준 진단 라벨셋인 MIMIC-IV-ECG-Ext-ICD**(PhysioNet)가 바로 이 방식을 씁니다.
각 ECG를 기록 시각으로 ED/입원 stay에 매칭하고(admit/discharge/death time 비교), 그 stay의
**discharge 진단**을 ECG 라벨로 사용합니다 - 고정 ±N일 윈도우를 쓰지 않습니다.
본 노트북은 `condition_occurrence`의 `condition_start_date`~`condition_end_date`(= 그 진단이 내려진
입원/내원의 admit~discharge 구간)에 ECG 촬영일이 들어가는지로 이 관점을 재현합니다.

**다른 방식과 비교 (트레이드오프)**
- **방문 단위(본 노트북)**: 진단 코드가 보통 discharge 시점에 찍히므로, 고정 ±1일 같은 짧은 윈도우는
  같은 입원 중의 실제 발작 ECG를 놓치기 쉽습니다. stay 구간 전체로 매칭하면 이를 방지해 **더 정확**합니다.
- **고정 짧은 윈도우(예: ±1일)**: 촬영 시점엔 밀착하지만, 위 이유로 양성 ECG를 과소 라벨(성능 하락).
- **넓은 윈도우(예: 31일~1년)**: detection이 아니라 **prediction/screening** task용
  (예: Attia et al. - sinus rhythm ECG로 미래 AF 예측, 발생 31일 전 이내 sinus ECG를 양성으로).
- **시점 무관(환자 병력)**: 미래 진단 누수가 생겨 "이 ECG가 AFib인가"와 어긋남.

**참고문헌**
- MIMIC-IV-ECG-Ext-ICD: Diagnostic labels for MIMIC-IV-ECG (PhysioNet)
  https://physionet.org/content/mimic-iv-ecg-ext-icd-labels/1.0.1/
- MIMIC-IV-ECG: Diagnostic Electrocardiogram Matched Subset (PhysioNet)
  https://physionet.org/content/mimic-iv-ecg/1.0/
- Attia et al., Lancet 2019 - AI-ECG identification of AF during sinus rhythm (prediction, 31일 윈도우 예시)
  https://pubmed.ncbi.nlm.nih.gov/31378392/
